In [ ]:
import keras
import tensorflow as tf
from preprocessing import train_ds, test_ds, valid_ds, TRAIN_SIZE, TEST_SIZE, VALID_SIZE

tf.random.set_seed(0)

he_init=keras.initializers.HeNormal()

elu_act=keras.activations.elu

l2_reg=keras.regularizers.l2(0.001)

inputs=keras.layers.Input(shape=(128,128,1), name='Input')

x=keras.layers.Conv2D(32, kernel_size=(5,5), kernel_initializer=he_init, padding='same', kernel_regularizer=l2_reg)(inputs)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
x=keras.layers.Conv2D(32, kernel_size=(3,3), kernel_initializer=he_init, padding='same', kernel_regularizer=l2_reg)(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
block_1_output=keras.layers.MaxPool2D((2,2))(x)

x=keras.layers.Conv2D(64, kernel_size=(3,3), kernel_initializer=he_init, padding='same', kernel_regularizer=l2_reg)(block_1_output)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
x=keras.layers.Conv2D(64, kernel_size=(3,3), kernel_initializer=he_init, padding='same', kernel_regularizer=l2_reg)(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
skip_layer_1=keras.layers.Conv2D(64, kernel_size=(1,1), kernel_initializer=he_init, padding='same', kernel_regularizer=l2_reg)(block_1_output)
x=keras.layers.add([x, skip_layer_1])
x=keras.layers.MaxPool2D((2,2))(x)
block_2_output=x

x=keras.layers.Conv2D(128, kernel_size=(3,3), kernel_initializer=he_init, padding='same', kernel_regularizer=l2_reg)(block_2_output)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
x=keras.layers.Conv2D(128, kernel_size=(3,3), kernel_initializer=he_init, padding='same', kernel_regularizer=l2_reg)(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
skip_layer_2=keras.layers.Conv2D(128, kernel_size=(1,1), kernel_initializer=he_init, padding='same', kernel_regularizer=l2_reg)(block_2_output)
x=keras.layers.add([x,skip_layer_2])
x=keras.layers.MaxPool2D((2,2))(x)
block_3_output=x

x=keras.layers.GlobalAveragePooling2D()(block_3_output)
x=keras.layers.Dense(128, activation=elu_act, kernel_initializer=he_init, kernel_regularizer=l2_reg)(x)
x=keras.layers.Dense(64, activation=elu_act, kernel_initializer=he_init, kernel_regularizer=l2_reg)(x)
outputs=keras.layers.Dense(14, activation='sigmoid')(x)

model=keras.Model(inputs=inputs, outputs=outputs)

Adam_optimizer=keras.optimizers.Adam(0.0001, clipnorm=1.0)

SGD_optimizer=keras.optimizers.SGD(learning_rate=0.001, momentum=0.95, nesterov=True)

loss=keras.losses.BinaryCrossentropy()

model.compile(optimizer=SGD_optimizer, loss=loss, metrics=[keras.metrics.AUC(multi_label=True,num_labels=14,name="auc")
                                                        ,keras.metrics.BinaryAccuracy(name="binary_accuracy")])

lr_plateau=keras.callbacks.ReduceLROnPlateau(monitor="val_auc",patience=2, mode="max", factor=0.5, min_lr=1e-5, verbose=True)

earlyStop_cb=keras.callbacks.EarlyStopping(monitor="val_auc", patience=4, restore_best_weights=True, mode="max", verbose=True)

history=model.fit(train_ds, validation_data=valid_ds, epochs=20, callbacks=[earlyStop_cb, lr_plateau], steps_per_epoch=TRAIN_SIZE//32, validation_steps=VALID_SIZE//32)

model.save("CNN_128_3Block_GAP_BN_DO02_L2_SGD_LrPlateau.keras")

2026-01-21 23:57:36.238701: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2026-01-21 23:57:36.238761: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-01-21 23:57:36.238766: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-01-21 23:57:36.238790: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-01-21 23:57:36.238804: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Epoch 1/20


2026-01-21 23:57:37.416060: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


  35/2452 ━━━━━━━━━━━━━━━━━━━━ 18:38 463ms/step - auc: 0.4885 - binary_accuracy: 0.7411 - loss: 2.3147

2026-01-21 23:57:55.907101: W tensorflow/core/lib/png/png_io.cc:89] PNG warning: iCCP: profile 'ICC Profile': 'GRAY': Gray color space not permitted on RGB PNG


...

2452/2452 ━━━━━━━━━━━━━━━━━━━━ 0s 959ms/step - auc: 0.5439 - binary_accuracy: 0.8814 - loss: 1.9077

2026-01-22 00:36:57.915308: W tensorflow/core/lib/png/png_io.cc:89] PNG warning: iCCP: profile 'ICC Profile': 'GRAY': Gray color space not permitted on RGB PNG
2026-01-22 00:38:54.654708: W tensorflow/core/lib/png/png_io.cc:89] PNG warning: iCCP: profile 'ICC Profile': 'GRAY': Gray color space not permitted on RGB PNG


2452/2452 ━━━━━━━━━━━━━━━━━━━━ 2666s 1s/step - auc: 0.5684 - binary_accuracy: 0.8866 - loss: 1.8166 - val_auc: 0.5315 - val_binary_accuracy: 0.9027 - val_loss: 1.6314 - learning_rate: 0.0010
Epoch 2/20


2026-01-22 00:42:03.258180: W tensorflow/core/kernels/data/cache_dataset_ops.cc:302] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


2452/2452 ━━━━━━━━━━━━━━━━━━━━ 1340s 546ms/step - auc: 0.5968 - binary_accuracy: 0.8877 - loss: 1.5416 - val_auc: 0.5377 - val_binary_accuracy: 0.9015 - val_loss: 1.3910 - learning_rate: 0.0010
Epoch 3/20
2452/2452 ━━━━━━━━━━━━━━━━━━━━ 2107s 859ms/step - auc: 0.6057 - binary_accuracy: 0.8879 - loss: 1.3195 - val_auc: 0.5360 - val_binary_accuracy: 0.9134 - val_loss: 1.1927 - learning_rate: 0.0010
Epoch 4/20
2452/2452 ━━━━━━━━━━━━━━━━━━━━ 1109s 452ms/step - auc: 0.6118 - binary_accuracy: 0.8878 - loss: 1.1373 - val_auc: 0.5625 - val_binary_accuracy: 0.9394 - val_loss: 1.0045 - learning_rate: 0.0010
Epoch 5/20
2452/2452 ━━━━━━━━━━━━━━━━━━━━ 1403s 572ms/step - auc: 0.6158 - binary_accuracy: 0.8879 - loss: 0.9874 - val_auc: 0.5653 - val_binary_accuracy: 0.9301 - val_loss: 0.8707 - learning_rate: 0.0010
Epoch 6/20
2452/2452 ━━━━━━━━━━━━━━━━━━━━ 1016s 414ms/step - auc: 0.6182 - binary_accuracy: 0.8880 - loss: 0.8645 - val_auc: 0.5841 - val_binary_accuracy: 0.9259 - val_loss: 0.7576 - learning

In [ ]:
import keras
import tensorflow as tf
from preprocessing import train_ds, test_ds, valid_ds, TRAIN_SIZE, TEST_SIZE, VALID_SIZE, weighted_bce

tf.random.set_seed(0)

he_init=keras.initializers.HeNormal()

elu_act=keras.activations.elu

l2_reg=keras.regularizers.l2(0.001)

data_aug=keras.Sequential([
    keras.layers.RandomFlip("horizontal"),
    keras.layers.RandomTranslation((0.05,0.05)),
])

inputs=keras.layers.Input(shape=(128,128,1), name='Input')

x=data_aug(inputs)

x=keras.layers.Conv2D(32, kernel_size=(5,5), kernel_initializer=he_init, padding='same', kernel_regularizer=l2_reg)(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
x=keras.layers.Conv2D(32, kernel_size=(3,3), kernel_initializer=he_init, padding='same', kernel_regularizer=l2_reg)(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
block_1_output=keras.layers.MaxPool2D((2,2))(x)

x=keras.layers.Conv2D(64, kernel_size=(3,3), kernel_initializer=he_init, padding='same', kernel_regularizer=l2_reg)(block_1_output)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
x=keras.layers.Conv2D(64, kernel_size=(3,3), kernel_initializer=he_init, padding='same', kernel_regularizer=l2_reg)(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
skip_layer_1=keras.layers.Conv2D(64, kernel_size=(1,1), kernel_initializer=he_init, padding='same', kernel_regularizer=l2_reg)(block_1_output)
x=keras.layers.add([x, skip_layer_1])
x=keras.layers.MaxPool2D((2,2))(x)
block_2_output=x

x=keras.layers.Conv2D(128, kernel_size=(3,3), kernel_initializer=he_init, padding='same', kernel_regularizer=l2_reg)(block_2_output)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
x=keras.layers.Conv2D(128, kernel_size=(3,3), kernel_initializer=he_init, padding='same', kernel_regularizer=l2_reg)(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
skip_layer_2=keras.layers.Conv2D(128, kernel_size=(1,1), kernel_initializer=he_init, padding='same', kernel_regularizer=l2_reg)(block_2_output)
x=keras.layers.add([x,skip_layer_2])
x=keras.layers.MaxPool2D((2,2))(x)
block_3_output=x

x=keras.layers.GlobalAveragePooling2D()(block_3_output)
x=keras.layers.Dense(128, activation=elu_act, kernel_initializer=he_init, kernel_regularizer=l2_reg)(x)
x=keras.layers.Dense(64, activation=elu_act, kernel_initializer=he_init, kernel_regularizer=l2_reg)(x)
outputs=keras.layers.Dense(14, activation='sigmoid')(x)

model=keras.Model(inputs=inputs, outputs=outputs)

Adam_optimizer=keras.optimizers.Adam(0.0001, clipnorm=1.0)

SGD_optimizer=keras.optimizers.SGD(learning_rate=0.0005, momentum=0.95, nesterov=True)

loss=keras.losses.BinaryCrossentropy()

model.compile(optimizer=Adam_optimizer, loss=weighted_bce, metrics=[keras.metrics.AUC(multi_label=True,num_labels=14,name="auc")
                                                        ,keras.metrics.BinaryAccuracy(name="binary_accuracy")])

lr_plateau=keras.callbacks.ReduceLROnPlateau(monitor="val_auc",patience=2, mode="max", factor=0.5, min_lr=1e-5, verbose=True)

earlyStop_cb=keras.callbacks.EarlyStopping(monitor="val_auc", patience=4, restore_best_weights=True, mode="max", verbose=True)

history=model.fit(train_ds, validation_data=valid_ds, epochs=10, callbacks=[earlyStop_cb], steps_per_epoch=TRAIN_SIZE//32, validation_steps=VALID_SIZE//32)

model.save("CNN_128_3Block_GAP_BN_DO02_L2_SGD_LrPlateau.keras")

ImportError: cannot import name 'weighted_bce' from 'preprocessing' (/Users/Vivaan/Documents/VS Code/Deep Learning/NIH/preprocessing.py)

In [ ]:
import keras
import tensorflow as tf
from preprocessing import train_ds, test_ds, valid_ds, TRAIN_SIZE, TEST_SIZE, VALID_SIZE, weighted_bce

tf.random.set_seed(0)

he_init=keras.initializers.HeNormal()

elu_act=keras.activations.elu

l2_reg=keras.regularizers.l2(0.001)

data_aug=keras.Sequential([
    keras.layers.RandomFlip("horizontal"),
    keras.layers.RandomTranslation((0.05,0.05)),
])

inputs=keras.layers.Input(shape=(128,128,1), name='Input')

x=data_aug(inputs)

x=keras.layers.Conv2D(32, kernel_size=(5,5), kernel_initializer=he_init, padding='same', kernel_regularizer=l2_reg)(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
x=keras.layers.Conv2D(32, kernel_size=(3,3), kernel_initializer=he_init, padding='same', kernel_regularizer=l2_reg)(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
block_1_output=keras.layers.MaxPool2D((2,2))(x)

x=keras.layers.Conv2D(64, kernel_size=(3,3), kernel_initializer=he_init, padding='same', kernel_regularizer=l2_reg)(block_1_output)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
x=keras.layers.Conv2D(64, kernel_size=(3,3), kernel_initializer=he_init, padding='same', kernel_regularizer=l2_reg)(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
skip_layer_1=keras.layers.Conv2D(64, kernel_size=(1,1), kernel_initializer=he_init, padding='same', kernel_regularizer=l2_reg)(block_1_output)
x=keras.layers.add([x, skip_layer_1])
x=keras.layers.MaxPool2D((2,2))(x)
block_2_output=x

x=keras.layers.Conv2D(128, kernel_size=(3,3), kernel_initializer=he_init, padding='same', kernel_regularizer=l2_reg)(block_2_output)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
x=keras.layers.Conv2D(128, kernel_size=(3,3), kernel_initializer=he_init, padding='same', kernel_regularizer=l2_reg)(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
skip_layer_2=keras.layers.Conv2D(128, kernel_size=(1,1), kernel_initializer=he_init, padding='same', kernel_regularizer=l2_reg)(block_2_output)
x=keras.layers.add([x,skip_layer_2])
x=keras.layers.MaxPool2D((2,2))(x)
block_3_output=x

x=keras.layers.GlobalAveragePooling2D()(block_3_output)
x=keras.layers.Dense(128, activation=elu_act, kernel_initializer=he_init, kernel_regularizer=l2_reg)(x)
x=keras.layers.Dense(64, activation=elu_act, kernel_initializer=he_init, kernel_regularizer=l2_reg)(x)
outputs=keras.layers.Dense(14, activation='sigmoid')(x)

model=keras.Model(inputs=inputs, outputs=outputs)

Adam_optimizer=keras.optimizers.Adam(0.0001, clipnorm=1.0)

SGD_optimizer=keras.optimizers.SGD(learning_rate=0.0005, momentum=0.95, nesterov=True)

loss=keras.losses.BinaryCrossentropy()

model.compile(optimizer=Adam_optimizer, loss=weighted_bce, metrics=[keras.metrics.AUC(multi_label=True,num_labels=14,name="auc")
                                                        ,keras.metrics.BinaryAccuracy(name="binary_accuracy")])

lr_plateau=keras.callbacks.ReduceLROnPlateau(monitor="val_auc",patience=2, mode="max", factor=0.5, min_lr=1e-5, verbose=True)

earlyStop_cb=keras.callbacks.EarlyStopping(monitor="val_auc", patience=4, restore_best_weights=True, mode="max", verbose=True)

history=model.fit(train_ds, validation_data=valid_ds, epochs=10, callbacks=[earlyStop_cb], steps_per_epoch=TRAIN_SIZE//32, validation_steps=VALID_SIZE//32)

model.save("CNN_128_3Block_GAP_BN_DO02_L2_SGD_LrPlateau.keras")

ImportError: cannot import name 'weighted_bce' from 'preprocessing' (/Users/Vivaan/Documents/VS Code/Deep Learning/NIH/preprocessing.py)

In [ ]:
import keras
import tensorflow as tf
from preprocessing import train_ds, test_ds, valid_ds, TRAIN_SIZE, TEST_SIZE, VALID_SIZE, weighted_bce

tf.random.set_seed(0)

he_init=keras.initializers.HeNormal()

elu_act=keras.activations.elu

l2_reg=keras.regularizers.l2(0.001)

data_aug=keras.Sequential([
    keras.layers.RandomFlip("horizontal"),
    keras.layers.RandomTranslation((0.05,0.05)),
])

inputs=keras.layers.Input(shape=(128,128,1), name='Input')

x=data_aug(inputs)

x=keras.layers.Conv2D(32, kernel_size=(5,5), kernel_initializer=he_init, padding='same', kernel_regularizer=l2_reg)(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
x=keras.layers.Conv2D(32, kernel_size=(3,3), kernel_initializer=he_init, padding='same', kernel_regularizer=l2_reg)(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
block_1_output=keras.layers.MaxPool2D((2,2))(x)

x=keras.layers.Conv2D(64, kernel_size=(3,3), kernel_initializer=he_init, padding='same', kernel_regularizer=l2_reg)(block_1_output)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
x=keras.layers.Conv2D(64, kernel_size=(3,3), kernel_initializer=he_init, padding='same', kernel_regularizer=l2_reg)(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
skip_layer_1=keras.layers.Conv2D(64, kernel_size=(1,1), kernel_initializer=he_init, padding='same', kernel_regularizer=l2_reg)(block_1_output)
x=keras.layers.add([x, skip_layer_1])
x=keras.layers.MaxPool2D((2,2))(x)
block_2_output=x

x=keras.layers.Conv2D(128, kernel_size=(3,3), kernel_initializer=he_init, padding='same', kernel_regularizer=l2_reg)(block_2_output)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
x=keras.layers.Conv2D(128, kernel_size=(3,3), kernel_initializer=he_init, padding='same', kernel_regularizer=l2_reg)(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
skip_layer_2=keras.layers.Conv2D(128, kernel_size=(1,1), kernel_initializer=he_init, padding='same', kernel_regularizer=l2_reg)(block_2_output)
x=keras.layers.add([x,skip_layer_2])
x=keras.layers.MaxPool2D((2,2))(x)
block_3_output=x

x=keras.layers.GlobalAveragePooling2D()(block_3_output)
x=keras.layers.Dense(128, activation=elu_act, kernel_initializer=he_init, kernel_regularizer=l2_reg)(x)
x=keras.layers.Dense(64, activation=elu_act, kernel_initializer=he_init, kernel_regularizer=l2_reg)(x)
outputs=keras.layers.Dense(14, activation='sigmoid')(x)

model=keras.Model(inputs=inputs, outputs=outputs)

Adam_optimizer=keras.optimizers.Adam(0.0001, clipnorm=1.0)

SGD_optimizer=keras.optimizers.SGD(learning_rate=0.0005, momentum=0.95, nesterov=True)

loss=keras.losses.BinaryCrossentropy()

model.compile(optimizer=Adam_optimizer, loss=weighted_bce, metrics=[keras.metrics.AUC(multi_label=True,num_labels=14,name="auc")
                                                        ,keras.metrics.BinaryAccuracy(name="binary_accuracy")])

lr_plateau=keras.callbacks.ReduceLROnPlateau(monitor="val_auc",patience=2, mode="max", factor=0.5, min_lr=1e-5, verbose=True)

earlyStop_cb=keras.callbacks.EarlyStopping(monitor="val_auc", patience=4, restore_best_weights=True, mode="max", verbose=True)

history=model.fit(train_ds, validation_data=valid_ds, epochs=10, callbacks=[earlyStop_cb], steps_per_epoch=TRAIN_SIZE//32, validation_steps=VALID_SIZE//32)

model.save("CNN_128_3Block_GAP_BN_DO02_L2_SGD_LrPlateau.keras")

2026-01-22 15:38:05.814794: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2026-01-22 15:38:05.814848: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-01-22 15:38:05.814852: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-01-22 15:38:05.815398: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-01-22 15:38:05.815416: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


TypeError: RandomTranslation.__init__() missing 1 required positional argument: 'width_factor'

In [ ]:
import keras
import tensorflow as tf
from preprocessing import train_ds, test_ds, valid_ds, TRAIN_SIZE, TEST_SIZE, VALID_SIZE, weighted_bce

tf.random.set_seed(0)

he_init=keras.initializers.HeNormal()

elu_act=keras.activations.elu

l2_reg=keras.regularizers.l2(0.001)

data_aug=keras.Sequential([
    keras.layers.RandomFlip("horizontal"),
    keras.layers.RandomTranslation(height_factor=0.05, width_factor=0.05),
])

inputs=keras.layers.Input(shape=(128,128,1), name='Input')

x=data_aug(inputs)

x=keras.layers.Conv2D(32, kernel_size=(5,5), kernel_initializer=he_init, padding='same', kernel_regularizer=l2_reg)(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
x=keras.layers.Conv2D(32, kernel_size=(3,3), kernel_initializer=he_init, padding='same', kernel_regularizer=l2_reg)(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
block_1_output=keras.layers.MaxPool2D((2,2))(x)

x=keras.layers.Conv2D(64, kernel_size=(3,3), kernel_initializer=he_init, padding='same', kernel_regularizer=l2_reg)(block_1_output)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
x=keras.layers.Conv2D(64, kernel_size=(3,3), kernel_initializer=he_init, padding='same', kernel_regularizer=l2_reg)(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
skip_layer_1=keras.layers.Conv2D(64, kernel_size=(1,1), kernel_initializer=he_init, padding='same', kernel_regularizer=l2_reg)(block_1_output)
x=keras.layers.add([x, skip_layer_1])
x=keras.layers.MaxPool2D((2,2))(x)
block_2_output=x

x=keras.layers.Conv2D(128, kernel_size=(3,3), kernel_initializer=he_init, padding='same', kernel_regularizer=l2_reg)(block_2_output)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
x=keras.layers.Conv2D(128, kernel_size=(3,3), kernel_initializer=he_init, padding='same', kernel_regularizer=l2_reg)(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
skip_layer_2=keras.layers.Conv2D(128, kernel_size=(1,1), kernel_initializer=he_init, padding='same', kernel_regularizer=l2_reg)(block_2_output)
x=keras.layers.add([x,skip_layer_2])
x=keras.layers.MaxPool2D((2,2))(x)
block_3_output=x

x=keras.layers.GlobalAveragePooling2D()(block_3_output)
x=keras.layers.Dense(128, activation=elu_act, kernel_initializer=he_init, kernel_regularizer=l2_reg)(x)
x=keras.layers.Dense(64, activation=elu_act, kernel_initializer=he_init, kernel_regularizer=l2_reg)(x)
outputs=keras.layers.Dense(14, activation='sigmoid')(x)

model=keras.Model(inputs=inputs, outputs=outputs)

Adam_optimizer=keras.optimizers.Adam(0.0001, clipnorm=1.0)

SGD_optimizer=keras.optimizers.SGD(learning_rate=0.0005, momentum=0.95, nesterov=True)

loss=keras.losses.BinaryCrossentropy()

model.compile(optimizer=Adam_optimizer, loss=weighted_bce, metrics=[keras.metrics.AUC(multi_label=True,num_labels=14,name="auc")
                                                        ,keras.metrics.BinaryAccuracy(name="binary_accuracy")])

lr_plateau=keras.callbacks.ReduceLROnPlateau(monitor="val_auc",patience=2, mode="max", factor=0.5, min_lr=1e-5, verbose=True)

earlyStop_cb=keras.callbacks.EarlyStopping(monitor="val_auc", patience=4, restore_best_weights=True, mode="max", verbose=True)

history=model.fit(train_ds, validation_data=valid_ds, epochs=10, callbacks=[earlyStop_cb], steps_per_epoch=TRAIN_SIZE//32, validation_steps=VALID_SIZE//32)

model.save("CNN_128_3Block_GAP_BN_DO02_L2_SGD_LrPlateau.keras")

Epoch 1/10


2026-01-22 15:39:52.456288: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


  80/2452 ━━━━━━━━━━━━━━━━━━━━ 17:41 447ms/step - auc: 0.5022 - binary_accuracy: 0.7439 - loss: 3.7490

2026-01-22 15:40:32.604124: W tensorflow/core/lib/png/png_io.cc:89] PNG warning: iCCP: profile 'ICC Profile': 'GRAY': Gray color space not permitted on RGB PNG


2452/2452 ━━━━━━━━━━━━━━━━━━━━ 0s 617ms/step - auc: 0.5606 - binary_accuracy: 0.6302 - loss: 2.9681

2026-01-22 16:05:08.896952: W tensorflow/core/kernels/data/cache_dataset_ops.cc:302] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.
2026-01-22 16:05:16.609717: W tensorflow/core/lib/png/png_io.cc:89] PNG warning: iCCP: profile 'ICC Profile': 'GRAY': Gray color space not permitted on RGB PNG
2026-01-22 16:06:49.283599: W tensorflow/core/lib/png/png_io.cc:89] PNG warning: iCCP: profile 'ICC Profile': 'GRAY': Gray color space not permitted on RGB PNG


2452/2452 ━━━━━━━━━━━━━━━━━━━━ 1740s 708ms/step - auc: 0.5842 - binary_accuracy: 0.6317 - loss: 2.7767 - val_auc: 0.5418 - val_binary_accuracy: 0.6872 - val_loss: 2.7963
Epoch 2/10


2026-01-22 16:08:51.677571: W tensorflow/core/kernels/data/cache_dataset_ops.cc:302] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


2452/2452 ━━━━━━━━━━━━━━━━━━━━ 3571s 1s/step - auc: 0.6188 - binary_accuracy: 0.6448 - loss: 2.3541 - val_auc: 0.5555 - val_binary_accuracy: 0.6075 - val_loss: 2.4575
Epoch 3/10
2452/2452 ━━━━━━━━━━━━━━━━━━━━ 1819s 742ms/step - auc: 0.6306 - binary_accuracy: 0.6431 - loss: 2.0960 - val_auc: 0.5559 - val_binary_accuracy: 0.6329 - val_loss: 2.2738
Epoch 4/10
2452/2452 ━━━━━━━━━━━━━━━━━━━━ 1867s 761ms/step - auc: 0.6395 - binary_accuracy: 0.6363 - loss: 1.9161 - val_auc: 0.5741 - val_binary_accuracy: 0.5092 - val_loss: 2.1734
Epoch 5/10
2452/2452 ━━━━━━━━━━━━━━━━━━━━ 3349s 1s/step - auc: 0.6473 - binary_accuracy: 0.6293 - loss: 1.7839 - val_auc: 0.6176 - val_binary_accuracy: 0.4331 - val_loss: 1.9929
Epoch 6/10
2452/2452 ━━━━━━━━━━━━━━━━━━━━ 1404s 573ms/step - auc: 0.6574 - binary_accuracy: 0.6299 - loss: 1.6795 - val_auc: 0.6367 - val_binary_accuracy: 0.3767 - val_loss: 1.8566
Epoch 7/10
2452/2452 ━━━━━━━━━━━━━━━━━━━━ 2016s 822ms/step - auc: 0.6686 - binary_accuracy: 0.6344 - loss: 1.593